# Raw inter-frame Δangle → Rayleigh noise

`D = hypot(Δφ, Δθ)` between adjacent Kerr samples. No saccade mask in the pickle.

## `density=True` on `ax.hist`

Default `hist` plots **counts**. Taller bins then just mean “more samples,” and the height scales with `n` and with bin width, so you cannot overlay a probability formula.

`density=True` rescales each bar so that **area = probability**. Bar height is
`(count in bin) / (n × bin width)`, and the bars integrate to 1. That is the same
vertical scale as the Rayleigh PDF, which is why the red/black curve can sit on the histogram.
It is **not** a percent. It is also **not** “fraction of samples in the bin” unless the bin width is 1.

## What B is (your teacher’s note)

Two independent Gaussians `Δφ, Δθ ~ N(0, σ)` give Euclidean distance
`D = sqrt(Δφ² + Δθ²) ~ Rayleigh(scale B)` with **B = σ** (so `Sigma = 1` → `Scale = 1`).
Fitting Rayleigh to `D` therefore estimates the **generating Gaussian σ on each axis**.
That is the noise measure: how large a typical frame-to-frame step is on φ or θ, in deg/frame.

A true Rayleigh also has skew ≈ **0.631** and excess kurtosis ≈ **0.245**, independent of B.
If those are far off, the sample is not a single Gaussian-noise process (saccades/glitches in the tail).

## Iterate per panel

Each condition has its own cell. Change **`HI_PCT`**, **`XMAX`**, **`N_BINS`**, optionally **`THRESHOLD`**, re-run that cell.
- `HI_PCT = 100` — fit and draw every sample (untrimmed).
- `HI_PCT = 99.5` — drop the top 0.5% of *that* condition, then fit (fast way to look at the core).
- `XMAX` — x-axis **and** an upper clip: histogram and Rayleigh both use samples ≤ `XMAX`. Lower it to inspect the core; `None` uses the max of the (percentile-clipped) sample.
- **SNR** of the trim is `D_cut / B`: `D_cut` is the largest D still in the fit (`HI_PCT`, then `XMAX` if tighter). `B` is the Rayleigh scale of that core (per-axis Gaussian σ). Same units, so the ratio is how many noise-σ sit between the core and the cutoff. If `THRESHOLD` is set, `threshold / B` is the detector SNR given this same B.

Pickle: `outputs/replotting_standalone/raw_interframe_delta/metadata/raw_interframe_delta.pkl`

In [ ]:
%matplotlib inline
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "eye_tracking_system_tools").is_dir())

PKL = REPO / "outputs/replotting_standalone/raw_interframe_delta/metadata/raw_interframe_delta.pkl"
with open(PKL, "rb") as f:
    data = pickle.load(f)

POOLS = {k: np.asarray(v, dtype=float) for k, v in data["pools"].items()}
POOLS = {k: v[np.isfinite(v) & (v >= 0)] for k, v in POOLS.items()}
META = {
    "rigid": ("rigid lizard", "#D55E00"),
    "modular": ("modular lizard", "#0072B2"),
    "turtle": ("turtle", "#CC79A7"),
    "mouse": ("mouse", "#009E73"),
}
SKEW_T, KURT_T = 0.631, 0.245
print({k: int(v.size) for k, v in POOLS.items()})


def clip_hi(a, hi_pct):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a) & (a >= 0)]
    if a.size == 0 or hi_pct >= 100:
        return a
    return a[a <= np.percentile(a, hi_pct)]


def rayleigh_fit(a):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a) & (a >= 0)]
    if a.size < 8:
        return {"n": int(a.size), "B": np.nan, "skew": np.nan, "kurt": np.nan}
    B = float(np.sqrt(np.mean(a * a) / 2.0))
    return {
        "n": int(a.size),
        "B": B,
        "skew": float(stats.skew(a, bias=False)),
        "kurt": float(stats.kurtosis(a, fisher=True, bias=False)),
    }


def plot_one(mount, *, hi_pct=100, xmax=None, n_bins=50, threshold=None):
    """One condition, one figure. Hist and Rayleigh fit use the same samples.

    SNR from the trim is D_cut / B: D_cut is the upper edge of the fitted sample
    (HI_PCT percentile, then XMAX if tighter) and B is the Rayleigh scale of that
    core. If THRESHOLD is set, snr_thr = threshold / B is the detector SNR given
    this noise estimate.
    """
    label, color = META[mount]
    raw = POOLS[mount]
    d_hi = (
        float(np.percentile(raw, hi_pct))
        if (raw.size and hi_pct < 100)
        else (float(raw.max()) if raw.size else np.nan)
    )
    a = clip_hi(raw, hi_pct)
    if xmax is not None:
        a = a[a <= float(xmax)]
    fit = rayleigh_fit(a)
    B = fit["B"]
    d_fit = float(a.max()) if a.size else np.nan
    snr = (d_fit / B) if (np.isfinite(B) and B > 0 and np.isfinite(d_fit)) else np.nan
    snr_thr = (
        (float(threshold) / B)
        if (threshold is not None and np.isfinite(B) and B > 0)
        else np.nan
    )
    fit = {**fit, "d_hi": d_hi, "d_fit": d_fit, "snr": snr, "snr_thr": snr_thr}
    hi = float(xmax) if xmax is not None else (float(a.max()) if a.size else 1.0)
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    if a.size:
        ax.hist(
            a, bins=n_bins, range=(0, hi), density=True,
            color=color, edgecolor="0.35", alpha=0.75, label="data",
        )
        if np.isfinite(B) and B > 0:
            xs = np.linspace(0, hi, 400)
            ax.plot(xs, (xs / B**2) * np.exp(-0.5 * (xs / B) ** 2), color="0.1", lw=1.8, label=f"Rayleigh B={B:.3f}")
        if np.isfinite(d_fit) and 0 < d_fit < hi:
            ax.axvline(d_fit, color="0.25", ls=":", lw=1.2, label=f"trim {d_fit:.3g}")
        if threshold is not None:
            ax.axvline(float(threshold), color="#D55E00", ls="--", lw=1.2, label=f"threshold {threshold:g}")
        bits = [f"{label}: B={B:.4g} deg/frame", f"D_cut={d_fit:.4g}", f"SNR=D_cut/B={snr:.2f}"]
        if np.isfinite(snr_thr):
            bits.append(f"threshold/B={snr_thr:.2f}")
        print("  ".join(bits))
    ax.set_xlim(0, hi)
    ax.set_xlabel("Distance D = hypot(Δφ, Δθ)  [deg/frame]")
    ax.set_ylabel("Probability density")
    clip = "untrimmed" if hi_pct >= 100 and xmax is None else f"≤ p{hi_pct:g}"
    snr_txt = f"SNR={snr:.2f}" if np.isfinite(snr) else "SNR=nan"
    ax.set_title(
        f"{label}  {clip}  n={fit['n']:,}  {snr_txt}\n"
        f"skew={fit['skew']:.3f} (want {SKEW_T})   kurt={fit['kurt']:.3f} (want {KURT_T})",
        fontsize=10,
    )
    ax.legend(fontsize=8, frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    return pd.Series(fit, name=mount), fig

## Choose `HI_PCT` from Rayleigh shape (skew / kurtosis)

A Rayleigh sample should sit at **skew ≈ 0.631** (left axis) and **excess kurtosis ≈ 0.245** (right axis), independent of noise scale B.

Sweep `HI_PCT` from 100 downward: each step drops more of the upper tail, refits shape, and plots both traces. Read **left → right as more trimming**.

- While the tail still dominates, both traces are huge.
- The **knee** is where they fall toward the dashed Rayleigh targets and flatten.
- If you trim too far you start cutting the noise core (truncated Rayleigh) and the traces can peel away from the targets again.

The vertical line is the `HI_PCT` that minimizes the normalized distance to `(0.631, 0.245)`. That value is stored in `SUGGESTED_HI_PCT` for the per-condition cells below. Edit `HI_MIN` / `HI_STEP` and re-run.

In [ ]:
# Sweep HI_PCT from 100 downward. Prefix of the sorted sample = percentile clip (no extra xmax).
HI_MIN = 70.0
HI_STEP = 0.5
HI_PCTS = np.arange(100.0, HI_MIN - 1e-9, -HI_STEP)


def sweep_hi_pct(mount: str, hi_pcts: np.ndarray = HI_PCTS) -> pd.DataFrame:
    sorted_a = np.sort(POOLS[mount])
    n = sorted_a.size
    rows = []
    for hi in hi_pcts:
        n_keep = max(8, int(np.floor(hi / 100.0 * n)))
        sub = sorted_a[:n_keep]
        fit = rayleigh_fit(sub)
        rows.append(
            {
                "hi_pct": float(hi),
                "n": int(n_keep),
                "B": fit["B"],
                "skew": fit["skew"],
                "kurtosis": fit["kurt"],
            }
        )
    return pd.DataFrame(rows)


def shape_error(skew: np.ndarray, kurt: np.ndarray) -> np.ndarray:
    """Normalized distance to Rayleigh (skew, kurtosis)."""
    return np.hypot((skew - SKEW_T) / SKEW_T, (kurt - KURT_T) / KURT_T)


def plot_hi_pct_sweep(mount: str, table: pd.DataFrame) -> float:
    err = shape_error(table["skew"].to_numpy(), table["kurtosis"].to_numpy())
    i_best = int(np.nanargmin(err))
    hi_best = float(table["hi_pct"].iloc[i_best])
    b_best = float(table["B"].iloc[i_best])
    sk_best = float(table["skew"].iloc[i_best])
    ku_best = float(table["kurtosis"].iloc[i_best])

    fig, ax_sk = plt.subplots(figsize=(9, 4.2))
    ax_ku = ax_sk.twinx()

    (ln_sk,) = ax_sk.plot(
        table["hi_pct"], table["skew"], color="#0072B2", lw=1.8, label="skew"
    )
    (ln_ku,) = ax_ku.plot(
        table["hi_pct"], table["kurtosis"], color="#D55E00", lw=1.8, label="kurtosis"
    )
    ax_sk.axhline(SKEW_T, color="#0072B2", ls="--", lw=1.0, alpha=0.85)
    ax_ku.axhline(KURT_T, color="#D55E00", ls="--", lw=1.0, alpha=0.85)
    ax_sk.axvline(hi_best, color="0.25", ls=":", lw=1.2)

    ax_sk.set_xlim(100.0, HI_MIN)
    ax_sk.set_xlabel("HI_PCT  (100 → more tail removed →)")
    ax_sk.set_ylabel("skew", color="#0072B2")
    ax_ku.set_ylabel("excess kurtosis", color="#D55E00")
    ax_sk.tick_params(axis="y", colors="#0072B2")
    ax_ku.tick_params(axis="y", colors="#D55E00")
    ax_sk.set_title(
        f"{mount}: Rayleigh-shape vs HI_PCT   "
        f"knee @ {hi_best:.1f}  (B={b_best:.4g}, skew={sk_best:.3f}, kurt={ku_best:.3f})"
    )
    ax_sk.legend([ln_sk, ln_ku], ["skew", "kurtosis"], loc="upper right", frameon=False)
    ax_sk.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

    print(
        f"{mount}: suggested HI_PCT={hi_best:.1f}  "
        f"n={int(table['n'].iloc[i_best]):,}  B={b_best:.4g}  "
        f"skew={sk_best:.3f} (target {SKEW_T:.3f})  "
        f"kurt={ku_best:.3f} (target {KURT_T:.3f})  "
        f"shape_err={err[i_best]:.3f}"
    )
    return hi_best


SUGGESTED_HI_PCT = {}
SWEEP_TABLES = {}
for _mount in ("rigid", "modular", "turtle", "mouse"):
    SWEEP_TABLES[_mount] = sweep_hi_pct(_mount)
    SUGGESTED_HI_PCT[_mount] = plot_hi_pct_sweep(_mount, SWEEP_TABLES[_mount])

print("\nSUGGESTED_HI_PCT =", {k: round(v, 1) for k, v in SUGGESTED_HI_PCT.items()})

### Rigid lizard
Paper detector is **0.8 deg/frame**. Median D on the raw pickle is ~0.06. Start with a tight `XMAX` so the core is visible; raise it if you want the tail.

`display(fit)` includes **`snr` = `D_cut / B`** from this `HI_PCT`/`XMAX` trim, and **`snr_thr` = threshold / B**.

In [ ]:
HI_PCT = 100
XMAX = 0.5
N_BINS = 50
THRESHOLD = 0.8

fit, fig = plot_one("rigid", hi_pct=HI_PCT, xmax=XMAX, n_bins=N_BINS, threshold=THRESHOLD)
display(fit)
print(f"trim SNR (D_cut/B) = {fit['snr']:.2f}    detector SNR (threshold/B) = {fit['snr_thr']:.2f}")
fig

### Modular lizard
Paper detector is **0.8 deg/frame**. `snr` is `D_cut / B` from this trim; `snr_thr` is `0.8 / B`.

In [ ]:
HI_PCT = 100
XMAX = 0.4
N_BINS = 50
THRESHOLD = 0.8

fit, fig = plot_one("modular", hi_pct=HI_PCT, xmax=XMAX, n_bins=N_BINS, threshold=THRESHOLD)
display(fit)
print(f"trim SNR (D_cut/B) = {fit['snr']:.2f}    detector SNR (threshold/B) = {fit['snr_thr']:.2f}")
fig

### Turtle
No paper detector here (`THRESHOLD = None`), so only trim SNR: **`snr` = `D_cut / B`**.

In [ ]:
HI_PCT = 100
XMAX = 1.5
N_BINS = 50
THRESHOLD = None

fit, fig = plot_one("turtle", hi_pct=HI_PCT, xmax=XMAX, n_bins=N_BINS, threshold=THRESHOLD)
display(fit)
_thr = fit["snr_thr"]
_thr_txt = "nan" if not np.isfinite(_thr) else f"{_thr:.2f}"
print(f"trim SNR (D_cut/B) = {fit['snr']:.2f}    detector SNR (threshold/B) = {_thr_txt}")
fig

### Mouse
GUI finalize thresholds on disk were about **2.5–3.9 deg/frame** (mean ~3.23). Median raw D is ~0.64.

`snr` is `D_cut / B` from this trim; `snr_thr` is `3.23 / B`.

In [ ]:
HI_PCT = 100
XMAX = 3.0
N_BINS = 50
THRESHOLD = 3.23

fit, fig = plot_one("mouse", hi_pct=HI_PCT, xmax=XMAX, n_bins=N_BINS, threshold=THRESHOLD)
display(fit)
print(f"trim SNR (D_cut/B) = {fit['snr']:.2f}    detector SNR (threshold/B) = {fit['snr_thr']:.2f}")
fig